In [1]:
%cd ..
%load_ext autoreload
%autoreload 2

/home/dongmin/userdata/dongmin/omr-data-preprocessing


/home/dongmin/userdata/dongmin/omr-data-preprocessing/.venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import os
import shutil
from pathlib import Path
from collections import defaultdict
from itertools import groupby

import math
import random

import json
import csv
import strictyaml as syaml

from tqdm import tqdm
import matplotlib.pyplot as plt

import cv2
import numpy as np
import music21
import musicdiff as md
import converter21 as c21

import xml.etree.ElementTree as ET
from modules.lmxe.lmxe import linearize_lmxe, delinearize_lmxe
from modules.lmxe.lmxe.utils import split_tokens_by_key_token, find_nth_token_index
from modules.lmxe.lmxe.Linearizer import LMXELinearizer
from modules.lmxe.lmxe.Delinearizer import LMXEDelinearizer
from modules.lmxe.lmxe.LMXEFile import LMXEFile, LMXEMetadata, get_score_type
from modules.lmxe.lmxe.symbolic.MxlFile import MusicXMLFile
from modules.lmxe.lmxe.symbolic.part_to_score import parts_to_score
from modules.lmxe.lmxe.symbolic.debug_compare import compare_parts

from omrdp.utils import load_ossq_metadata
from omrdp.utils import parse_log_and_get_exclusions, print_exclusion_report
from omrdp.utils.image import crop_white_space

from IPython.core.magic import register_cell_magic
@register_cell_magic
def skip(line, cell):
  return

## Load metadata

In [3]:
dataset_dir = Path.home() / 'userdata' / 'open-score-string-quartets'
dataset_data_dir = dataset_dir / 'data'
dataset_score_dir = dataset_dir / 'scores'

dataset_dir.exists()

True

In [4]:
metadata = load_ossq_metadata(dataset_data_dir / 'scores.yaml')
len(metadata)

122

In [5]:
temp_dir = Path.cwd() / 'temp'
temp_dir.mkdir(exist_ok=True)

## MusicXML to krn

In [6]:
fatal_error_set = {
  'conversion error',
  'measure insert/delete', 
  'staff insert/delete',
  'staff group',
  'barline',
  'keysig', 'clef', 'tie',
  'note', 'dot',
}

### Synthetic

In [7]:
krn_log_file = '/home/dongmin/userdata/open-score-string-quartets/logs/musicxml_to_krn_synthetic_2026-04-03-12:53:16.log'

ekrn_log_file = '/home/dongmin/userdata/open-score-string-quartets/logs/krn_to_ekrn_synthetic_2026-04-03-13:33:50.log'

excluded_segments_path = dataset_dir / 'excluded_segments_synthetic_krn_systemwise.txt'

In [8]:
krn_log_file = Path(krn_log_file)
(
  krn_excluded_segments, 
  krn_num_samples, 
  krn_known_errors, 
  krn_known_fatal_errors, 
  krn_error_cnt, 
  krn_failed_cnt
) = parse_log_and_get_exclusions(
  krn_log_file,
  metadata,
  dataset_score_dir,
  fatal_error_set=fatal_error_set,
)

print(f"KRN Conversion Exclusion Report:")
print_exclusion_report(
  num_samples=krn_num_samples,
  excluded_segments=krn_excluded_segments,
  known_errors=krn_known_errors,
  known_fatal_errors=krn_known_fatal_errors,
  error_cnt=krn_error_cnt,
  failed_cnt=krn_failed_cnt,
)

KRN Conversion Exclusion Report:
total: 13262
excluded segments: 361
exclusion rate: 2.722063037249284

error segments: 361
failed segments: 0

Known fatal errors:
{
  "conversion error": 21,
  "note": 18206,
  "measure insert/delete": 16581,
  "dot": 2482,
  "clef": 1126,
  "keysig": 3261,
  "tie": 78,
  "staff insert/delete": 9402,
  "staff group": 84
}

Known errors:
{
  "conversion error": 21,
  "slur": 9782,
  "flag/beam": 7773,
  "note": 18206,
  "ornament": 641,
  "measure insert/delete": 16581,
  "dot": 2482,
  "clef": 1126,
  "keysig": 3261,
  "tie": 78,
  "note head": 712,
  "fingered tremolo": 809,
  "accidental": 155,
  "staff insert/delete": 9402,
  "staff group": 84,
  "articulation": 10,
  "direction": 114
}


In [9]:
ekrn_log_file = Path(ekrn_log_file)
(
  ekrn_excluded_segments, 
  ekrn_num_samples, 
  ekrn_known_errors, 
  ekrn_known_fatal_errors, 
  ekrn_error_cnt, 
  ekrn_failed_cnt
) = parse_log_and_get_exclusions(
  ekrn_log_file,
  metadata,
  dataset_score_dir,
  fatal_error_set=fatal_error_set,
)

print(f"EKRN Conversion Exclusion Report:")
print_exclusion_report(
  num_samples=ekrn_num_samples,
  excluded_segments=ekrn_excluded_segments,
  known_errors=ekrn_known_errors,
  known_fatal_errors=ekrn_known_fatal_errors,
  error_cnt=ekrn_error_cnt,
  failed_cnt=ekrn_failed_cnt,
)

EKRN Conversion Exclusion Report:
total: 13239
excluded segments: 412
exclusion rate: 3.112017523982174

error segments: 412
failed segments: 0

Known fatal errors:
{
  "tie": 663,
  "note": 42,
  "measure insert/delete": 51
}

Known errors:
{
  "flag/beam": 425159,
  "tie": 663,
  "note": 42,
  "articulation": 16,
  "measure insert/delete": 51
}


In [10]:
krn_excluded_set = set(krn_excluded_segments)
ekrn_excluded_set = set(ekrn_excluded_segments)

print("len kern:", len(krn_excluded_set))
print("len ekern:", len(ekrn_excluded_set))
print("len intersection:", len(krn_excluded_set & ekrn_excluded_set))
print("len union:", len(krn_excluded_set | ekrn_excluded_set))
print("len krn - ekrn:", len(krn_excluded_set - ekrn_excluded_set))
print("len ekrn - krn:", len(ekrn_excluded_set - krn_excluded_set))

len kern: 340
len ekern: 412
len intersection: 17
len union: 735
len krn - ekrn: 323
len ekrn - krn: 395


In [11]:
excluded_list = list(krn_excluded_set | ekrn_excluded_set)
print(f"Total Segments: {krn_num_samples}")
print(f"Total Excluded Segments: {len(excluded_list)}")
print(f"Total Exclusion Rate: {len(excluded_list) / krn_num_samples * 100:.2f}%")

Total Segments: 13262
Total Excluded Segments: 735
Total Exclusion Rate: 5.54%


In [12]:
with open(excluded_segments_path, 'w') as f:
  for seg_id in excluded_list:
    f.write(seg_id + '\n')

### Scanned

In [13]:
krn_log_file = '/home/dongmin/userdata/open-score-string-quartets/logs/musicxml_to_krn_scanned_2026-04-03-13:17:01.log'

ekrn_log_file = '/home/dongmin/userdata/open-score-string-quartets/logs/krn_to_ekrn_scanned_2026-04-03-13:40:11.log'

excluded_segments_path = dataset_dir / 'excluded_segments_scanned_krn_systemwise.txt'

In [14]:
krn_log_file = Path(krn_log_file)
(
  krn_excluded_segments, 
  krn_num_samples, 
  krn_known_errors, 
  krn_known_fatal_errors, 
  krn_error_cnt, 
  krn_failed_cnt
) = parse_log_and_get_exclusions(
  krn_log_file,
  metadata,
  dataset_score_dir,
  fatal_error_set=fatal_error_set,
)

print(f"KRN Conversion Exclusion Report:")
print_exclusion_report(
  excluded_segments=krn_excluded_segments,
  num_samples=krn_num_samples,
  known_errors=krn_known_errors,
  known_fatal_errors=krn_known_fatal_errors,
  error_cnt=krn_error_cnt,
  failed_cnt=krn_failed_cnt,
)

KRN Conversion Exclusion Report:
total: 11324
excluded segments: 297
exclusion rate: 2.6227481455316144

error segments: 297
failed segments: 0

Known fatal errors:
{
  "conversion error": 19,
  "measure insert/delete": 15711,
  "note": 18370,
  "dot": 1837,
  "clef": 928,
  "keysig": 2626,
  "tie": 72,
  "staff insert/delete": 6460,
  "staff group": 60
}

Known errors:
{
  "conversion error": 19,
  "slur": 8206,
  "ornament": 580,
  "flag/beam": 7377,
  "measure insert/delete": 15711,
  "note": 18370,
  "dot": 1837,
  "clef": 928,
  "keysig": 2626,
  "tie": 72,
  "note head": 654,
  "fingered tremolo": 734,
  "accidental": 144,
  "staff insert/delete": 6460,
  "staff group": 60,
  "articulation": 13,
  "direction": 76
}


In [15]:
ekrn_log_file = Path(ekrn_log_file)
(
  ekrn_excluded_segments, 
  ekrn_num_samples, 
  ekrn_known_errors, 
  ekrn_known_fatal_errors, 
  ekrn_error_cnt, 
  ekrn_failed_cnt
) = parse_log_and_get_exclusions(
  ekrn_log_file,
  metadata,
  dataset_score_dir,
  fatal_error_set=fatal_error_set,
)

print(f"EKRN Conversion Exclusion Report:")
print_exclusion_report(
  excluded_segments=ekrn_excluded_segments,
  num_samples=ekrn_num_samples,
  known_errors=ekrn_known_errors,
  known_fatal_errors=ekrn_known_fatal_errors,
  error_cnt=ekrn_error_cnt,
  failed_cnt=ekrn_failed_cnt,
)

EKRN Conversion Exclusion Report:
total: 11303
excluded segments: 347
exclusion rate: 3.069981420861718

error segments: 347
failed segments: 0

Known fatal errors:
{
  "tie": 572,
  "note": 37,
  "measure insert/delete": 51
}

Known errors:
{
  "flag/beam": 338665,
  "tie": 572,
  "note": 37,
  "articulation": 16,
  "measure insert/delete": 51
}


In [16]:
krn_excluded_set = set(krn_excluded_segments)
ekrn_excluded_set = set(ekrn_excluded_segments)

print("len kern:", len(krn_excluded_set))
print("len ekern:", len(ekrn_excluded_set))
print("len intersection:", len(krn_excluded_set & ekrn_excluded_set))
print("len union:", len(krn_excluded_set | ekrn_excluded_set))
print("len krn - ekrn:", len(krn_excluded_set - ekrn_excluded_set))
print("len ekrn - krn:", len(ekrn_excluded_set - krn_excluded_set))

len kern: 278
len ekern: 347
len intersection: 12
len union: 613
len krn - ekrn: 266
len ekrn - krn: 335


In [17]:
excluded_list = list(krn_excluded_set | ekrn_excluded_set)
print(f"Total Segments: {krn_num_samples}")
print(f"Total Excluded Segments: {len(excluded_list)}")
print(f"Total Exclusion Rate: {len(excluded_list) / krn_num_samples * 100:.2f}%")

Total Segments: 11324
Total Excluded Segments: 613
Total Exclusion Rate: 5.41%


In [18]:
excluded_list = list(krn_excluded_set | ekrn_excluded_set)

with open(excluded_segments_path, 'w') as f:
  for seg_id in excluded_list:
    f.write(seg_id + '\n')